# Price Preprocess Exploration

Interactive notebook for manually inspecting every step in `pystocks.preprocess.price`.

What this notebook covers:
- raw price-history loading from SQLite
- full preprocess execution with configurable thresholds
- per-step flag inspection: invalid prices, stale runs, outlier returns, bridge anomalies
- eligibility diagnostics and manual chart review for individual `conid`s


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR if (NOTEBOOK_DIR / "pystocks").exists() else NOTEBOOK_DIR.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from pystocks.config import SQLITE_DB_PATH
from pystocks.preprocess.price import (
    PricePreprocessConfig,
    _mark_price_level_anomalies,
    _mark_stale_rows,
    _numeric_series,
    _robust_outlier_mask,
    load_price_history,
    preprocess_price_history,
)

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 200)
DEFAULT_SQLITE_PATH = SQLITE_DB_PATH
print(f"Notebook directory: {NOTEBOOK_DIR}")
print(f"Project root: {PROJECT_ROOT}")
print(f"SQLite path: {DEFAULT_SQLITE_PATH}")


In [ ]:
# Adjust these before rerunning the notebook.
sqlite_path = DEFAULT_SQLITE_PATH
sample_conids = []

config = PricePreprocessConfig(
    min_history_days=252,
    max_missing_ratio=0.30,
    max_internal_gap_days=20,
    stale_run_max_days=5,
    outlier_z_threshold=50.0,
    local_price_ratio_threshold=5.0,
    bridge_outlier_span_max_rows=5,
)

config


In [ ]:
raw_prices = load_price_history(sqlite_path=sqlite_path)
print(f"Loaded {len(raw_prices):,} price rows across {raw_prices['conid'].nunique():,} conids")
display(raw_prices.head())

if sample_conids:
    working_prices = raw_prices.loc[raw_prices["conid"].astype(str).isin([str(v) for v in sample_conids])].copy()
else:
    chosen = raw_prices["conid"].astype(str).drop_duplicates().sort_values()
    working_prices = raw_prices.loc[raw_prices["conid"].astype(str).isin(chosen)].copy()

print(f"Working set: {len(working_prices):,} rows across {working_prices['conid'].nunique():,} conids")
display(working_prices.groupby("conid").agg(rows=("trade_date", "size"), start=("trade_date", "min"), end=("trade_date", "max")).reset_index())


In [ ]:
result = preprocess_price_history(working_prices, config=config, show_progress=False)
prices = result["prices"].copy()
eligibility = result["eligibility"].copy()

print("Eligibility summary")
display(eligibility)

print("Flag rates")
display(
    pd.DataFrame(
        {
            "metric": [
                "rows",
                "valid_price_rows",
                "stale_rows",
                "outlier_rows",
                "price_level_anomaly_rows",
                "clean_rows",
            ],
            "value": [
                len(prices),
                int(prices["is_valid_price"].sum()),
                int(prices["is_stale_price"].sum()),
                int(prices["is_outlier_return"].sum()),
                int(prices["is_price_level_anomaly"].sum()),
                int(prices["is_clean_price"].sum()),
            ],
        }
    )
)


In [ ]:
def build_stage_frame(price_df: pd.DataFrame, conid: str, cfg: PricePreprocessConfig) -> pd.DataFrame:
    group = price_df.loc[price_df["conid"].astype(str) == str(conid)].copy()
    if group.empty:
        raise ValueError(f"No rows found for conid={conid}")

    group = group.sort_values("trade_date").reset_index(drop=True)
    group["price_value"] = (
        group["close"]
        .where(group["close"].notna(), group["price"])
        .where(group["close"].notna() | group["price"].notna(), group["open"])
    )

    high_values = _numeric_series(group, "high")
    low_values = _numeric_series(group, "low")
    group["is_valid_price"] = (
        group["price_value"].notna()
        & np.isfinite(group["price_value"])
        & (group["price_value"] > 0)
        & ~(high_values.notna() & low_values.notna() & (low_values > high_values))
    )

    group["raw_return"] = group["price_value"].pct_change(fill_method=None)
    group["is_stale_price"] = _mark_stale_rows(group, cfg.stale_run_max_days)

    group["candidate_return"] = group["raw_return"].where(
        group["is_valid_price"] & ~group["is_stale_price"]
    )
    group["is_outlier_return"] = _robust_outlier_mask(
        group["candidate_return"],
        cfg.outlier_z_threshold,
    )

    base_clean_mask = (
        group["is_valid_price"] & ~group["is_stale_price"] & ~group["is_outlier_return"]
    )
    group["is_price_level_anomaly"] = _mark_price_level_anomalies(
        group,
        base_clean_mask,
        group["is_outlier_return"],
        cfg.local_price_ratio_threshold,
        cfg.bridge_outlier_span_max_rows,
    )
    group["is_clean_price"] = base_clean_mask & ~group["is_price_level_anomaly"]
    group["clean_price"] = group["price_value"].where(group["is_clean_price"])
    group["clean_return"] = group["clean_price"].pct_change(fill_method=None)

    return group


def stage_summary(stage_df: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame(
        {
            "metric": [
                "rows",
                "valid_price_rows",
                "stale_rows",
                "outlier_rows",
                "price_level_anomaly_rows",
                "clean_rows",
            ],
            "value": [
                len(stage_df),
                int(stage_df["is_valid_price"].sum()),
                int(stage_df["is_stale_price"].sum()),
                int(stage_df["is_outlier_return"].sum()),
                int(stage_df["is_price_level_anomaly"].sum()),
                int(stage_df["is_clean_price"].sum()),
            ],
        }
    )


def plot_stage_review(stage_df: pd.DataFrame, title: str | None = None) -> None:
    fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True, constrained_layout=True)

    axes[0].plot(stage_df["trade_date"], stage_df["price_value"], label="price_value", color="steelblue", linewidth=1.5)
    axes[0].plot(stage_df["trade_date"], stage_df["clean_price"], label="clean_price", color="black", linewidth=1.5)

    flag_specs = [
        ("is_valid_price", False, "invalid", "tab:red", "x"),
        ("is_stale_price", True, "stale", "tab:orange", "o"),
        ("is_outlier_return", True, "outlier", "tab:purple", "s"),
        ("is_price_level_anomaly", True, "anomaly", "tab:green", "D"),
    ]
    for column, positive_value, label, color, marker in flag_specs:
        mask = stage_df[column] == positive_value
        if mask.any():
            axes[0].scatter(
                stage_df.loc[mask, "trade_date"],
                stage_df.loc[mask, "price_value"],
                label=label,
                color=color,
                marker=marker,
                s=45,
            )

    axes[0].set_ylabel("Price")
    axes[0].set_title(title or "Price preprocess review")
    axes[0].legend(loc="best")

    axes[1].plot(stage_df["trade_date"], stage_df["raw_return"], label="raw_return", color="tab:blue", alpha=0.7)
    axes[1].plot(stage_df["trade_date"], stage_df["clean_return"], label="clean_return", color="black", alpha=0.9)
    if stage_df["is_outlier_return"].any():
        mask = stage_df["is_outlier_return"]
        axes[1].scatter(
            stage_df.loc[mask, "trade_date"],
            stage_df.loc[mask, "raw_return"],
            label="outlier return",
            color="tab:purple",
            marker="s",
            s=45,
        )
    axes[1].axhline(0.0, color="grey", linewidth=1)
    axes[1].set_ylabel("Return")
    axes[1].legend(loc="best")
    plt.show()


def review_conid(conid: str) -> pd.DataFrame:
    stage_df = build_stage_frame(working_prices, conid=str(conid), cfg=config)
    elig_row = eligibility.loc[eligibility["conid"].astype(str) == str(conid)]
    print(f"conid={conid}")
    if not elig_row.empty:
        display(elig_row)
    display(stage_summary(stage_df))
    plot_stage_review(stage_df, title=f"Price preprocess review: conid={conid}")
    display(
        stage_df.loc[
            ~stage_df["is_valid_price"]
            | stage_df["is_stale_price"]
            | stage_df["is_outlier_return"]
            | stage_df["is_price_level_anomaly"],
            [
                "trade_date",
                "price",
                "open",
                "high",
                "low",
                "close",
                "price_value",
                "raw_return",
                "clean_return",
                "is_valid_price",
                "is_stale_price",
                "is_outlier_return",
                "is_price_level_anomaly",
                "is_clean_price",
            ],
        ].sort_values("trade_date")
    )
    return stage_df


In [ ]:
display(
    prices.groupby("conid").agg(
        rows=("trade_date", "size"),
        invalid_rows=("is_valid_price", lambda s: int((~s).sum())),
        stale_rows=("is_stale_price", "sum"),
        outlier_rows=("is_outlier_return", "sum"),
        anomaly_rows=("is_price_level_anomaly", "sum"),
        clean_rows=("is_clean_price", "sum"),
    ).reset_index().sort_values(["anomaly_rows", "outlier_rows", "stale_rows", "invalid_rows"], ascending=False)
)


In [ ]:
# Pick any conid from the working set and rerun this cell.
conid_to_review = str(eligibility.iloc[0]["conid"]) if not eligibility.empty else None
stage_df = review_conid(conid_to_review) if conid_to_review is not None else pd.DataFrame()


In [ ]:
# Convenience queries for targeted manual review.
print("Ineligible conids")
display(eligibility.loc[~eligibility["eligible"]].sort_values(["eligibility_reason", "conid"]))

print("Rows removed by each step")
display(prices.loc[~prices["is_valid_price"] | prices["is_stale_price"] | prices["is_outlier_return"] | prices["is_price_level_anomaly"]].sort_values(["conid", "trade_date"]))


## Notes

- To inspect the full universe, set `sample_conids = []` and increase `sample_size`, or replace `working_prices` with `raw_prices` in the preprocess call.
- The notebook intentionally imports the module's internal helpers so you can audit each stage with the exact production logic.
- If you want to test threshold sensitivity, change `config` and rerun from the load/preprocess cells downward.
